# 🏗️ Graph State — Four Ways to Define State

## Learning Objectives
In this notebook, you will learn:
1. **Dictionary state** — the most basic, schema-free approach
2. **TypedDict state** — lightweight typing with custom reducers
3. **Dataclass state** — default values alongside type hints
4. **Pydantic state** — full validation with `Field()` and `BaseModel`

## Prerequisites
- `langgraph`, `pydantic` installed
- Familiarity with Python typing (`TypedDict`, `Annotated`)

---
## 🔧 Part 1: Environment Setup

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
from typing import TypedDict, List, Annotated
from dataclasses import dataclass

from pydantic import Field, BaseModel
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

print("✅ Imports loaded successfully!")

✅ Imports loaded successfully!


---
## ⚙️ Part 2: Shared Nodes & Helper

We define two nodes and a reusable `build_and_run_graph` helper so we can test
each state schema with the same graph topology.

In [2]:
# ============================================================================
# SHARED NODES: node_a and node_b
# ============================================================================
def node_a(state):
    """A simple node that updates the state."""
    print("Executing Node A...")
    return {
        "messages": ["Step A Completed"],
        "step_count": 1
    }


def node_b(state):
    """A simple node that updates the state."""
    print("Executing Node B...")
    if isinstance(state, dict):
        step_count = state["step_count"]
    else:
        step_count = state.step_count

    print(f"Current step count from state: {step_count}")
    return {
        "messages": ["Step B Completed"],
        "step_count": 1
    }

print("✅ Shared nodes defined!")

✅ Shared nodes defined!


In [3]:
# ============================================================================
# HELPER: Build and run a graph with any state schema
# ============================================================================
def build_and_run_graph(state_schema, initial_state):
    schema_name = state_schema.__name__ if hasattr(state_schema, "__name__") else "Dictionary"
    print(f"\n--- Building and running graph with state schema: {schema_name}")

    graph = StateGraph(state_schema)

    graph.add_node("node_a", node_a)
    graph.add_node("node_b", node_b)

    graph.add_edge(START, "node_a")
    graph.add_edge("node_a", "node_b")
    graph.add_edge("node_b", END)

    agent = graph.compile()
    final_state = agent.invoke(initial_state)

    print("\nFinal State:")
    print(final_state)
    print("-" * 40)

print("✅ Helper function defined!")

✅ Helper function defined!


---
## 📦 Part 3: Method 1 — Simple Dictionary

The most basic approach: no schema, no validation, no custom reducers.
State is just a plain `dict`. Quick for prototyping, but lacks type safety.

### Key Concepts:
- **No type safety** — any key can be added or misspelled
- **Default reducer only** — values get replaced, not merged

In [5]:
# ============================================================================
# METHOD 1: Dictionary State
# ============================================================================
def create_dict_state():
    return {
        "messages": [],
        "step_count": 0,
        "private_data": None
    }

build_and_run_graph(dict, create_dict_state())


--- Building and running graph with state schema: dict
Executing Node A...
Executing Node B...
Current step count from state: 1

Final State:
{'messages': ['Step B Completed'], 'step_count': 1}
----------------------------------------


---
## 🏷️ Part 4: Method 2 — TypedDict

TypedDict adds type annotations and allows attaching **custom reducers** via
`Annotated`. This is the most common approach in LangGraph.

### Key Concepts:
- **`Annotated[int, custom_add]`** — attaches a reducer that sums values
- **`Annotated[List[str], add_messages]`** — appends messages instead of replacing

In [6]:
# ============================================================================
# METHOD 2: TypedDict State with Custom Reducer
# ============================================================================
def custom_add(current: int, new: int):
    """A custom reducer that adds the new value to the current value."""
    return current + new


class TypedDictState(TypedDict):
    messages: Annotated[List[str], add_messages]
    step_count: Annotated[int, custom_add]
    private_data: str

build_and_run_graph(TypedDictState, {
    "messages": [],
    "step_count": 0,
    "private_data": ""
})


--- Building and running graph with state schema: TypedDictState
Executing Node A...
Executing Node B...
Current step count from state: 1

Final State:
{'messages': [HumanMessage(content='Step A Completed', additional_kwargs={}, response_metadata={}, id='08a0c225-1c8f-4598-9a03-e4502d194323'), HumanMessage(content='Step B Completed', additional_kwargs={}, response_metadata={}, id='26ac2e4e-e6df-452d-b896-ce6f5d436bd0')], 'step_count': 2, 'private_data': ''}
----------------------------------------


---
## 🗂️ Part 5: Method 3 — Dataclass

Dataclasses provide **default values** alongside type hints. More verbose than
TypedDict, but defaults simplify initialization.

### Key Concepts:
- **Default values** — `private_data: str = ""` avoids requiring it at init time
- **Same reducer support** via `Annotated`

In [8]:
# ============================================================================
# METHOD 3: Dataclass State
# ============================================================================
@dataclass
class DataClassState:
    messages: Annotated[List[str], add_messages]
    step_count: Annotated[int, custom_add]
    private_data: str = ""

build_and_run_graph(DataClassState, DataClassState(
    messages=[],
    step_count=0
))


--- Building and running graph with state schema: DataClassState
Executing Node A...
Executing Node B...
Current step count from state: 1

Final State:
{'messages': [HumanMessage(content='Step A Completed', additional_kwargs={}, response_metadata={}, id='f0f1ae05-69cd-42fa-be1b-7d4a7fadb609'), HumanMessage(content='Step B Completed', additional_kwargs={}, response_metadata={}, id='e9a4a26c-598e-47bc-b530-5f19d45ebc23')], 'step_count': 2, 'private_data': ''}
----------------------------------------


---
## 🛡️ Part 6: Method 4 — Pydantic Model

Pydantic provides the richest schema: **validation**, **default factories**
for mutable types, and full `BaseModel` features.

### Key Concepts:
- **`Field(default_factory=list)`** — safely creates a new list for each instance
- **`Field(default=0)`** — provides immutable defaults
- **Automatic validation** at runtime

In [9]:
# ============================================================================
# METHOD 4: Pydantic Model State
# ============================================================================
class PydanticState(BaseModel):
    messages: Annotated[List[BaseMessage], add_messages] = Field(default_factory=list)
    step_count: Annotated[int, custom_add] = Field(default=0)
    private_data: str = Field(default="")

build_and_run_graph(PydanticState, PydanticState())


--- Building and running graph with state schema: PydanticState
Executing Node A...
Executing Node B...
Current step count from state: 1

Final State:
{'messages': [HumanMessage(content='Step A Completed', additional_kwargs={}, response_metadata={}, id='345b1bac-5a8d-4616-a298-1f6dcb98317c'), HumanMessage(content='Step B Completed', additional_kwargs={}, response_metadata={}, id='eaf68e16-6068-45de-b019-9781e342c8f3')], 'step_count': 2, 'private_data': ''}
----------------------------------------


---
## 📝 Summary

In this notebook, we learned:

### 1. Four State Definition Methods
| Method | Type Safety | Default Values | Validation | Custom Reducers |
|--------|:-----------:|:--------------:|:----------:|:---------------:|
| `dict` | ❌ | ❌ | ❌ | ❌ |
| `TypedDict` | ✅ | ❌ | ❌ | ✅ |
| `@dataclass` | ✅ | ✅ | ❌ | ✅ |
| `BaseModel` | ✅ | ✅ | ✅ | ✅ |

### 2. Custom Reducers
- Attach via `Annotated[Type, reducer_fn]`
- Control how partial updates are merged into existing state

### Next Steps
- Deep dive into **Reducers** (notebook `04-reducers`)
- Explore **add_messages** with LLMs (notebook `05-add_messages`)